# Step 4 (Kaggle): river network and flood travel time

About 30–40 minutes. Settings: **GPU T4 x2** (or GPU T4), **Internet On**, then **Save Version → Save & Run All (Commit)**.

1. Build the river graph between CAMELS-IND gauges from the catchment boundaries (which gauge lies upstream of which).
2. Estimate the flood-wave travel time on each link (lag of the best flow correlation, and the delay between matched annual flood peaks).
3. Train LSTM_ALL_UP (3 seeds): Step 3's best LSTM plus the flow of the gauges directly upstream as an extra input. Compared with Step 3, on all gauges and on the gauges that have an upstream gauge.

In [ ]:
import subprocess, sys, os

LOG = '/kaggle/working/results/run_log.txt'
os.makedirs('/kaggle/working/results', exist_ok=True)

def run(cmd):
    """Run a shell command, show its output live, save it to the log, and stop if it fails."""
    print(f'\n$ {cmd}', flush=True)
    with open(LOG, 'a') as log:
        log.write(f'\n$ {cmd}\n')
        p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line)
        p.wait()
    if p.returncode != 0:
        raise RuntimeError(f'Step failed: {cmd} (see the output above or results/run_log.txt)')

run('nvidia-smi -L || echo no GPU')
run('cd /kaggle/working && (test -d flood || git clone -q https://github.com/yatharth7115/flood.git) && cd flood && git pull -q && git log --oneline -1')
os.chdir('/kaggle/working/flood')
run('pip install -q pytorch-forecasting pyyaml tabulate geopandas')

In [ ]:
import yaml
cfg = yaml.safe_load(open('configs/step2.yaml'))
cfg['data']['data_dir'] = '/tmp/camels_data'
cfg['data']['processed_file'] = '/tmp/camels_data/processed.parquet'
cfg['results_dir'] = '/kaggle/working/results'
os.makedirs('/tmp/camels_data', exist_ok=True)
yaml.safe_dump(cfg, open('configs/run.yaml', 'w'))
ALL_UP = 'antecedent cn future_rain upstream'

## 1. Data and river graph

In [ ]:
run('python src/download.py configs/run.yaml')
run('python src/dataset.py configs/run.yaml')
run('python src/river_graph.py configs/run.yaml')

## 2. LSTM with upstream flow, 3 seeds

In [ ]:
for s in range(3):
    run(f'python src/train_lstm.py configs/run.yaml --name LSTM_ALL_UP_S{s} --extra {ALL_UP} --peak-weight 4 --seed {s}')

## 3. Compare with Step 3

In [ ]:
R = cfg['results_dir']
run(f'python src/evaluate.py configs/run.yaml --reference results/step3/metrics_per_gauge.csv --ref-tag STEP3 '
    f'--base LSTM_ALL_ENS_STEP3 --subset {R}/gauges_with_upstream.csv --subset-tag with_upstream --plot LSTM_ALL_UP_ENS')
from IPython.display import Image, display, Markdown
display(Markdown(open(f'{R}/river_graph.md').read()[:6000]))
display(Markdown(open(f'{R}/summary.md').read()))
run(f'cd {R} && rm -f *_model.pt')